In [ ]:
!pip install roboflow ultralytics

In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from roboflow import Roboflow
from ultralytics import YOLO

import cv2
import os
import matplotlib.pyplot as plt

In [ ]:

rf = Roboflow(api_key="Q8B9HaFal1zOLYy0ovrW")
project = rf.workspace("licence-plate-3jqkb").project("license-plate-az47f")
version = project.version(1)
dataset = version.download("yolov12")


In [ ]:
print(dataset.location)

In [ ]:
!cp -r "/content/License-Plate-1" "/content/drive/MyDrive/Deep Learning Techniques and computer vision/Object Detection/"

In [ ]:
import os

directory_path = "/content/drive/MyDrive/Deep Learning Techniques and computer vision/Object Detection/"
os.makedirs(directory_path, exist_ok=True)
os.listdir(directory_path)

In [ ]:
import os

DATASET_PATH = "/content/drive/MyDrive/Deep Learning Techniques and computer vision/Object Detection/License-Plate-1"

print(os.path.exists(DATASET_PATH + "/data.yaml"))

In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/Deep Learning Techniques and computer vision/Object Detection"
DATASET_PATH = f"{PROJECT_PATH}/License-Plate-1"
RESULTS_PATH = f"{PROJECT_PATH}/results"

In [ ]:
print(PROJECT_PATH)
print(DATASET_PATH)
print(RESULTS_PATH)

In [ ]:
model = YOLO("yolo12n.pt")

In [ ]:
results = model.train(
    data=f"{DATASET_PATH}/data.yaml",
    epochs=10,
    imgsz=640,
    batch=16,
    project=RESULTS_PATH,
    name="exp1",
    plots=True
)

In [ ]:
valid_result = model.val()

In [ ]:
print("mAP50-95 :", valid_result.box.map)
print("mAP50    :", valid_result.box.map50)
print("mAP75    :", valid_result.box.map75)
print("Per Class:", valid_result.box.maps)

In [ ]:
valid_result.box.map50

In [ ]:
import matplotlib.pyplot as plt

img = plt.imread(f"{RESULTS_PATH}/exp1-2/results.png")

plt.figure(figsize=(12,8))
plt.imshow(img)
plt.axis("off")

In [ ]:
img = plt.imread(f"{RESULTS_PATH}/exp1-2/confusion_matrix.png")

plt.figure(figsize=(8,8))
plt.imshow(img)
plt.axis("off")

In [ ]:
model = YOLO(f"{RESULTS_PATH}/exp1-2/weights/best.pt")

In [ ]:
results = model.predict(
    source=f"{DATASET_PATH}/test/images",
    save=True,
    conf=0.25
)

In [ ]:
import glob
from IPython.display import Image, display

for img in glob.glob("runs/detect/predict/*.jpg")[:10]:
    display(Image(filename=img))

In [ ]:
VIDEO_PATH = "/content/drive/MyDrive/ Deep Learning Techniques and computer vision/Object Detection/video.mp4"

In [ ]:
import os

print(os.path.exists(VIDEO_PATH))

In [ ]:
from ultralytics import YOLO

car_model = YOLO("yolo12n.pt")

In [ ]:
car_model.track(
    source=VIDEO_PATH,
    tracker="bytetrack.yaml",
    persist=True,
    save=True,
    conf=0.4
)

In [ ]:
from pathlib import Path

for f in Path("runs").rglob("*.avi"):
    print(f)

for f in Path("runs").rglob("*.mp4"):
    print(f)

In [ ]:
!ffmpeg -i "runs/detect/runs/track_test/video.avi" \
-vcodec libx264 -acodec aac \
"runs/detect/runs/track_test/video.mp4" -y

In [ ]:
from IPython.display import Video

Video("runs/detect/runs/track_test/video.mp4", embed=True)

In [ ]:
from ultralytics import YOLO

plate_model = YOLO(f"{RESULTS_PATH}/exp1-2/weights/best.pt")
car_model = YOLO("yolo12n.pt")

In [ ]:
import os
print(os.path.exists(VIDEO_PATH))

In [ ]:
plate_model.predict(
    source=VIDEO_PATH,
    save=True,
    conf=0.25
)

In [ ]:
from pathlib import Path

for f in Path("runs").rglob("*"):
    print(f)

In [ ]:
!ffmpeg -i "runs/detect/predict/video.avi" \
-vcodec libx264 -acodec aac \
"runs/detect/predict/video.mp4" -y

In [ ]:
from IPython.display import Video

Video("runs/detect/predict/video.mp4", embed=True)

In [ ]:
car_results = car_model.track(
    source=VIDEO_PATH,
    tracker="bytetrack.yaml",
    stream=True,
    persist=True,
    conf=0.4
)

In [ ]:
for result in car_results:
    frame = result.orig_img.copy()

    if result.boxes is None:
        continue

    print("عدد العربيات:", len(result.boxes))
    break

In [ ]:
for result in car_results:
    frame = result.orig_img.copy()

    if result.boxes is None:
        continue

    for box in result.boxes:

        x1, y1, x2, y2 = map(int, box.xyxy[0])

        print(x1, y1, x2, y2)

    break

In [ ]:
import cv2
import math
car_results = car_model.track(
    source=VIDEO_PATH,
    tracker="bytetrack.yaml",
    stream=True,
    persist=True,
    conf=0.4
)
car_positions = {}
fps = 30
scale = 0.05
output_path = "final_result.mp4"
writer = None
for result in car_results:
    frame = result.orig_img.copy()
    if writer is None:
        h, w = frame.shape[:2]
        writer = cv2.VideoWriter(
            output_path,
            cv2.VideoWriter_fourcc(*"mp4v"),
            fps,
            (w, h)
        )

    if result.boxes is None:
        writer.write(frame)
        continue
    for box in result.boxes:
        cls = int(box.cls[0])
        if cls not in [2, 5, 7]:
            continue

        x1, y1, x2, y2 = map(int, box.xyxy[0])

        track_id = int(box.id.item()) if box.id is not None else -1

        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2

        speed = 0
        if track_id in car_positions:
            old_x, old_y = car_positions[track_id]
            distance_pixels = math.sqrt(
                (cx - old_x) ** 2 +
                (cy - old_y) ** 2
            )
            distance_meters = distance_pixels * scale
            speed = distance_meters * fps * 3.6
        car_positions[track_id] = (cx, cy)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            frame,
            f"Car {track_id}",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )
        cv2.putText(
            frame,
            f"{speed:.1f} km/h",
            (x1, y1 + 20),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 0, 0),
            2
        )
        car_crop = frame[y1:y2, x1:x2]
        plate_results = plate_model.predict(
            car_crop,
            conf=0.15,
            verbose=False
        )

        if len(plate_results[0].boxes) > 0:
            for plate in plate_results[0].boxes:

                px1, py1, px2, py2 = map(int, plate.xyxy[0])

                px1 += x1
                px2 += x1
                py1 += y1
                py2 += y1
                cv2.rectangle(
                    frame,
                    (px1, py1),
                    (px2, py2),
                    (0, 0, 255),
                    2
                )
    writer.write(frame)
writer.release()
print("Done")
print(output_path)

In [ ]:
!ffmpeg -i final_result.mp4 \
-vcodec libx264 -acodec aac \
final_result_h264.mp4 -y

In [ ]:
from IPython.display import Video
Video("final_result_h264.mp4", embed=True)